<a href="https://colab.research.google.com/github/EkaterinaLavlinskaya/ai-career-log/blob/main/logger_buttons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ========== ЛОГГЕР С КНОПКАМИ (обновлён) ==========
# Добавлена кнопка для удалённых вакансий

!pip install PyGithub > /dev/null 2>&1

from github import Github
import pandas as pd
from datetime import datetime
import json
import io
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# ========== НАСТРОЙКИ ==========
REPO_NAME = "EkaterinaLavlinskaya/ai-career-log"
FILE_PATH = "data/interactions.jsonl"
# ================================

# Подключаемся к GitHub
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("✅ Токен загружен из секретов Colab.")
except:
    print("❌ Не удалось загрузить токен.")
    raise

g = Github(GITHUB_TOKEN)

try:
    repo = g.get_repo(REPO_NAME)
    print(f"✅ Подключились к репозиторию: {REPO_NAME}")
except:
    print(f"❌ Ошибка: не могу найти репозиторий {REPO_NAME}")
    raise

# ========== ФУНКЦИИ ==========

def get_current_data(g):
    repo = g.get_repo(REPO_NAME)
    contents = repo.get_contents(FILE_PATH)
    content = contents.decoded_content.decode('utf-8')
    df = pd.read_json(io.StringIO(content), lines=True)
    return df, contents.sha

def add_record(g, new_data):
    repo = g.get_repo(REPO_NAME)
    contents = repo.get_contents(FILE_PATH)
    current_content = contents.decoded_content.decode('utf-8')
    current_sha = contents.sha

    df = pd.read_json(io.StringIO(current_content), lines=True)
    last = df.iloc[-1].to_dict()

    record = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "session_id": last.get("session_id", "001"),
        "topic": new_data.get("topic", "update"),
        "projects": new_data.get("projects", last["projects"]),
        "certificates": new_data.get("certificates", last["certificates"]),
        "days_continuous": new_data.get("days_continuous", last["days_continuous"]),
        "applications": new_data.get("applications", last["applications"]),
        "tests": new_data.get("tests", last["tests"]),
        "interviews": new_data.get("interviews", last.get("interviews", 0)),
        "offers": new_data.get("offers", last.get("offers", 0)),
        "rejections": new_data.get("rejections", last.get("rejections", 0)),
        "closed_vacancy": new_data.get("closed_vacancy", last.get("closed_vacancy", 0)),
        "review": new_data.get("review", last.get("review", 0)),
        "note": new_data.get("note", "")
    }

    new_line = json.dumps(record, ensure_ascii=False)
    updated_content = current_content + "\n" + new_line

    repo.update_file(
        path=FILE_PATH,
        message=f"Добавлена запись: {record['topic']}",
        content=updated_content.encode('utf-8'),
        sha=current_sha
    )
    return record

# ========== КНОПКИ ==========

note_input = widgets.Text(
    value='',
    placeholder='короткое описание (необязательно)',
    description='📝',
    layout=widgets.Layout(width='100%')
)

def make_button(desc, topic, color, default_note="", increment=None):
    btn = widgets.Button(description=desc, button_style=color, layout=widgets.Layout(width='160px'))

    def on_click(b):
        with out:
            clear_output(wait=True)
            display(note_input)
            display(widgets.HBox(button_row1))
            display(widgets.HBox(button_row2))
            display(widgets.HBox(button_row3))
            display(widgets.HBox(button_row4))
            try:
                note = note_input.value if note_input.value else default_note
                payload = {"topic": topic, "note": note}
                if increment:
                    payload.update(increment)
                rec = add_record(g, payload)
                print(f"✅ {desc}: {rec['note']}")
                print(f"   время: {rec['timestamp']}")
                note_input.value = ''
            except Exception as e:
                print(f"❌ Ошибка: {e}")
    btn.on_click(on_click)
    return btn

# Строки кнопок
button_row1 = [
    make_button('📁 Проект', 'project', 'success'),
    make_button('📬 Отклик', 'application', 'info'),
    make_button('🧪 Тест', 'test', 'warning'),
    make_button('🎓 Сертификат', 'certificate', 'primary')
]

button_row2 = [
    make_button('🤝 Собеседование', 'interview', 'danger'),
    make_button('🎉 Оффер', 'offer', 'success'),
    make_button('✅ День', 'daily', '', 'очередной день')
]

button_row3 = [
    make_button('❌ Отказ', 'rejection', 'danger'),
    make_button('📋 На рассмотрении', 'review', 'warning'),
    make_button('🗑️ Вакансия удалена', 'closed_vacancy', '')
]

button_row4 = [
    make_button('📝 Заметка', 'note', '')
]

out = widgets.Output()

# Отображаем
display(note_input)
display(widgets.HBox(button_row1))
display(widgets.HBox(button_row2))
display(widgets.HBox(button_row3))
display(widgets.HBox(button_row4))
display(out)

✅ Токен загружен из секретов Colab.
✅ Подключились к репозиторию: EkaterinaLavlinskaya/ai-career-log


/tmp/ipykernel_4372/3111513336.py:28: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  g = Github(GITHUB_TOKEN)


Text(value='', description='📝', layout=Layout(width='100%'), placeholder='короткое описание (необязательно)')

Output()